# Using peptideBERT

In [ ]:
#https://github.com/ChakradharG/PeptideBERT

Environment = protbert

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd

# Load the tokenizer and model from the repo/model hub
model_name = "ChakradharG/PeptideBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Example list of peptides
peptides = [
    "ACDEFGHIKLMNPQRSTVWY",  # 20 standard amino acids
    "MTEITAAMVKELRESTGAGMMDCKNALSETQHEWAY",
    "GHIJKLMNPQRSTVWYACDEF"
]

# Batch tokenize the peptides
inputs = tokenizer(peptides, padding=True, truncation=True, return_tensors="pt")

# Run the model to get the embeddings (last hidden states)
with torch.no_grad():
    outputs = model(**inputs)

# The last hidden state has shape (batch_size, sequence_length, hidden_size)
last_hidden_states = outputs.last_hidden_state

# Perform mean pooling on the token embeddings for each peptide
# You may want to exclude special tokens if needed. For simplicity, this example averages all tokens.
embeddings = last_hidden_states.mean(dim=1)  # shape: (batch_size, hidden_size)

# Convert embeddings to numpy arrays
embeddings_np = embeddings.detach().cpu().numpy()

# Create a pandas DataFrame to store the peptides and their corresponding embeddings.
df = pd.DataFrame({
    "peptide": peptides,
    "embedding": list(embeddings_np)  # each row is a numpy array of the embedding
})

# Optionally, if you want to save each dimension as a separate column, you can do:
hidden_size = embeddings_np.shape[1]
embedding_columns = {f"emb_{i}": embeddings_np[:, i] for i in range(hidden_size)}
df_expanded = pd.DataFrame({"peptide": peptides})
df_expanded = pd.concat([df_expanded, pd.DataFrame(embedding_columns)], axis=1)

print("DataFrame with embeddings as lists:")
print(df.head())
print("\nDataFrame with expanded embedding columns:")
print(df_expanded.head())

# Save the DataFrame to a CSV file if desired
df_expanded.to_csv("peptide_embeddings.csv", index=False)
